# Deep Learning Foundations Review

Run a small end-to-end classification workflow: create data, split it, train an MLP, select by validation loss, compare to a baseline, and evaluate test data once.

In [ ]:
import torch
from torch import nn

torch.manual_seed(42)
torch.set_num_threads(1)
print('CPU | end-to-end review workflow')

In [ ]:
features = torch.randn(600, 2)
labels = (features[:, 0] * features[:, 1] > 0).long()
train_x, train_y = features[:400], labels[:400]
valid_x, valid_y = features[400:500], labels[400:500]
test_x, test_y = features[500:], labels[500:]
assert (len(train_x), len(valid_x), len(test_x)) == (400, 100, 100)
print('split sizes:', len(train_x), len(valid_x), len(test_x))

In [ ]:
model = nn.Sequential(nn.Linear(2, 16), nn.Tanh(), nn.Linear(16, 2))
optimizer = torch.optim.Adam(model.parameters(), lr=0.02)
best_loss, best_state = float('inf'), None
for _ in range(100):
    optimizer.zero_grad()
    loss = nn.functional.cross_entropy(model(train_x), train_y)
    loss.backward()
    optimizer.step()
    with torch.no_grad():
        validation_loss = nn.functional.cross_entropy(model(valid_x), valid_y).item()
    if validation_loss < best_loss:
        best_loss = validation_loss
        best_state = {name: value.clone() for name, value in model.state_dict().items()}
model.load_state_dict(best_state)
with torch.no_grad():
    test_accuracy = (model(test_x).argmax(1) == test_y).float().mean().item()
    majority_accuracy = (test_y == train_y.bincount().argmax()).float().mean().item()
assert 0 <= test_accuracy <= 1
print(f'best validation loss={best_loss:.4f} | test accuracy={test_accuracy:.1%} | majority baseline={majority_accuracy:.1%}')